In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pylab as plt
from astropy.io import fits
from sklearn.cluster import KMeans
import sys
import os
import lightkurve as lk
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("star_functions.py"), '..')))
import star_functions as nana
import matplotlib.patches as mpatches

In [ ]:
fn = "/Users/natsuki/Projects/hogg_research/hoggnation/oscillator_catalog/good_parents_fit.fits"
with fits.open(fn) as hdu_list:
    print(hdu_list.info())
    data = hdu_list[1].data
    header = hdu_list[1].header
print(len(data), header)

In [ ]:
features = data["features"]
print(features.shape)
kics = data["star_id"]
print(kics.shape)

#create the time invariant feature vector for clustering (log(a0^2), log(a1^2 + b1^2), ...)
squared_feats = np.zeros((1438, 33))
for e,f in enumerate(features):
    j = 0
    for i in range(len(f) - 1):
        if i == 0:
            squared_feats[e][j] = f[i]**2
            j = j + 1
        if i%2 == 1:
            squared_feats[e][j] = f[i]**2 + f[i+1]**2
            j = j + 1

foo, k = squared_feats.shape
print(squared_feats.shape)
frequencies = np.outer(data["refined_frequency"], (1. + np.arange(k)))
informations = np.nansum(squared_feats * frequencies * (frequencies < 24.), axis=1)
print(informations)


refined_freqs = data["refined_frequency"]
good_mask = ~np.any(np.isnan(squared_feats), axis=1)
print("good mask", good_mask.shape)
print("squared feats before", squared_feats.shape)
squared_feats = squared_feats[good_mask]

print(squared_feats.shape)
log_squared_feats = np.log10(squared_feats)
log_squared_feats = log_squared_feats[:, 1:] #get rid of constant?

#syms = ["o", "X", "s", "D", "+", "*", "p", "^", "d"]
sizes = [3, 7, 11, 15, 18, 21]
sizes2 = [13,18,23,28]
sizes3 = [7, 11, 15, 17]
symbols = ["o", "^", "s", "p", "P", "*", "X", "D", "d"]

kics = kics[good_mask]
refined_freqs = refined_freqs[good_mask]
features = features[good_mask]
print(kics.shape, refined_freqs.shape)

print(np.all(squared_feats))

print(np.sum(squared_feats == 0))        # exact zeros
print(np.sum(squared_feats < 1e-300))    # near-zero values that log10 explodes
print(np.min(squared_feats))             # what is the smallest value?

print(log_squared_feats[:10])

In [ ]:
def phase_folding_by_rank(group, k, random_restart, kmeans, rank=0):
    labels = kmeans.labels_
    centroids = kmeans.cluster_centers_
    
    mask = labels == group
    true_indices = np.where(mask)[0]
    dists = np.linalg.norm(log_squared_feats[true_indices] - centroids[group], axis=1)
    sorted_indices = true_indices[np.argsort(dists)]
    if rank >= len(sorted_indices):
        print(f"rank {rank} out of range, only {len(sorted_indices)} stars in group")
        return
    i = sorted_indices[rank]
    
    star_id, feature, freq = kics[i], features[i], refined_freqs[i]
    #freq = 0.6299412024490911
    print(f"rank: {rank}, index: {i}, star_id: {star_id}, freq: {freq:.4f}")

    lc, delta_f, sampling_time, exptime = nana.get_kepler_data(star_id)
    t_fit, flux_fit, weight_fit = nana.mask_vals(lc, star_id)
   
    over_sampling = 3
    df, f_maxC = delta_f/over_sampling, (3 / (2*sampling_time))
    f_min = over_sampling * df
    freq_full, power_full = nana.get_periodogram(f_min, f_maxC, df, lc, star_id)
    conn = nana.get_db_connection()
    cursor = conn.cursor()
    cursor.execute(f"SELECT parent_frequency FROM parent_modes WHERE star_id = '{star_id}'")
    parent_freqs = [row[0] for row in cursor.fetchall()]
    cursor.execute(f"SELECT frequency FROM mode WHERE star_id = '{star_id}' AND parent_mode_id IS NOT NULL")
    child_freqs = [row[0] for row in cursor.fetchall()]
    conn.close()
    plt.plot(freq_full, power_full, 'k.', markersize=1, alpha=0.5)
    plt.xlabel("Frequency (1/day)")
    plt.ylabel("Power")
    plt.semilogy()
    plt.title(f"Log Periodogram of {star_id}")
    for pf in parent_freqs:
        plt.axvline(pf, color='red', alpha=0.6, lw=1)
    for cf in child_freqs:
        plt.axvline(cf, color='red', alpha=0.2, lw=0.5)
    plt.show()

    M = (len(feature) - 1) // 2
    print(f"feature length: {len(feature)}, M: {M}")

    theta = np.linspace(0, 4*np.pi, 1000)
    yplot0 = np.zeros_like(theta)
    
    for m in range(1, M + 1):
        a = feature[2*m - 1]
        b = feature[2*m]
        yplot0 += a * np.cos(m * theta) + b * np.sin(m * theta)
    yplot0+=feature[0]
        
    omega = 2 * np.pi * freq
    phase = (omega * t_fit) % (4 * np.pi)
    yplot = flux_fit
    plt.figure()
    plt.plot(phase, yplot, alpha=0.1, markersize=2, marker='.', linestyle='none', color='k')
    plt.plot(theta, yplot0, color='r', alpha=0.6, zorder=3)
    plt.xlabel('Phase')
    plt.ylabel('Flux')
    plt.title(rf'"b": Phase fold (radian) at freq {freq:.04f} $day^{{-1}}$ for {star_id}, group{group}')
    #if group == 6:
        #plt.savefig('b_g6i5_2026_06_08.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
##keep random number seem the same here. FIXED AT 82
K, r = 9, 82

kmeans = KMeans(n_clusters=K, init='k-means++', random_state=r)
kmeans.fit(log_squared_feats)
for group in range(K):
    phase_folding_by_rank(group, K, r, kmeans, 4)

In [ ]:
#final
K = 9
randoms = [82] #fixed random seed of 82

for r in randoms:
    kmeans = KMeans(n_clusters=K, init='k-means++', random_state=r)
    kmeans.fit(log_squared_feats)
    labels = kmeans.labels_
    centroids = kmeans.cluster_centers_

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    
    for ax, (i, j) in zip(axes, [(0, 1), (0, 2)]):
        for group in range(K):
            mask = labels == group
            sym = symbols[group % len(symbols)]
            size = sizes2[group % len(sizes2)]
            sc = ax.scatter(log_squared_feats[mask, i], log_squared_feats[mask, j],
                        c=[group] * mask.sum(),
                        vmin=0, vmax=8,
                        marker=sym,
                        s=size,
                        cmap='Set1',
                        linewidths=0.3,
                        edgecolors='black')
        #ax.scatter(centroids[:, i], centroids[:, j], c='red', s=25, alpha=0.8, marker='o')
        xmin = log_squared_feats[:, 0].min()
        ymin = min(log_squared_feats[:, 1].min(), log_squared_feats[:, 2].min())

        # then inside the ax loop:
        ax.set_ylim(-9.5, 0)
        ax.set_xlim(-7, 0)
        ax.set_xlabel(f"log_10 scalar {i+1}")
        ax.set_ylabel(f"log_10 scalar {j+1}")
        ax.grid(True, alpha=0.3)
        
    
    fig.suptitle("Amplitude - Cluster Plot")
    plt.tight_layout()
    plt.savefig('amp_cluster.png', bbox_inches='tight', dpi=150)
    plt.show()

## phase fold by kic id

In [ ]:
def phase_folding_by_kic(star_id, M=32):
    star_id = star_id.strip()
    
    # get frequency and features from arrays
    # mask = np.array([s.strip() == star_id for s in kics])
    # if not np.any(mask):
    #     print(f"star_id {star_id} not found in kics")
    #     return
    # i = np.where(mask)[0][0]
    feature, freq = features[i], refined_freqs[i]
    freq, feature = nana.fit_fourier_series_to_mode(27295, M) #TEMP, CHANGE THIS
    
    print(f"index: {i}, star_id: {star_id}, freq: {freq:.4f}")

    # get light curve
    lc, delta_f, sampling_time, exptime = nana.get_kepler_data(star_id)
    t_fit, flux_fit, weight_fit = nana.mask_vals(lc, star_id)

    # periodogram
    over_sampling = 3
    df, f_maxC = delta_f / over_sampling, (3 / (2 * sampling_time))
    f_min = over_sampling * df
    freq_full, power_full = nana.get_periodogram(f_min, f_maxC, df, lc, star_id)

    # get parent and child freqs from db
    conn = nana.get_db_connection()
    cursor = conn.cursor()
    cursor.execute(f"SELECT parent_frequency FROM parent_modes WHERE star_id = '{star_id}'")
    parent_freqs = [row[0] for row in cursor.fetchall()]
    cursor.execute(f"SELECT frequency FROM mode WHERE star_id = '{star_id}' AND parent_mode_id IS NOT NULL")
    child_freqs = [row[0] for row in cursor.fetchall()]
    conn.close()

    
    
    #lightcurve
    plt.figure(figsize=(5,5))
    plt.plot(t_fit, flux_fit, 'k-', linewidth=0.5, alpha=0.7)
    plt.xlabel("Time (days)")
    plt.ylabel("Flux")
    plt.title(f"Light curve of {star_id}")
    plt.tight_layout()
    plt.savefig('poster_child_lc_2026_06_08.png', bbox_inches='tight', dpi=150)
    plt.show()
        
    # periodogram plot
    plt.figure()
    plt.plot(freq_full, power_full, 'k.', markersize=1, alpha=0.5)
    plt.xlabel("Frequency (1/day)")
    plt.ylabel("Power")
    plt.semilogy()
    plt.title(f"Log Periodogram of {star_id}")
    for pf in parent_freqs:
        plt.axvline(pf, color='red', alpha=0.6, lw=1)
    for cf in child_freqs:
        plt.axvline(cf, color='red', alpha=0.2, lw=0.5)
    plt.savefig('poster_child_perio_2026_06_08.png', bbox_inches='tight', dpi=150)
    plt.show()

    # phase fold
    M = (len(feature) - 1) // 2
    print(f"feature length: {len(feature)}, M: {M}")
    theta = np.linspace(0, 4 * np.pi, 1000)
    yplot0 = np.zeros_like(theta)
    for m in range(1, M + 1):
        a = feature[2*m - 1]
        b = feature[2*m]
        yplot0 += a * np.cos(m * theta) + b * np.sin(m * theta)
    yplot0 += feature[0]

    omega = 2 * np.pi * freq
    phase = (omega * t_fit) % (4 * np.pi)

    plt.figure()
    plt.plot(phase, flux_fit, alpha=0.1, markersize=2, marker='.', linestyle='none', color='k')
    plt.plot(theta, yplot0, color='r', alpha=0.6, zorder=3)
    plt.xlabel('Phase')
    plt.ylabel('Flux')
    #plt.title(rf'Phase fold (radian) at freq {freq:.04f} $day^{{-1}}$ for {star_id}')
    plt.title(rf'Phase fold (radian) at freq {freq:.04f} $day^{{-1}}$ for {star_id}')
    plt.savefig('poster_child_phase_fold_2026_06_08.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
def phase_folding_by_kic(star_id, M=32):
    star_id = star_id.strip()
    
    feature, freq = features[i], refined_freqs[i]
    freq, feature = nana.fit_fourier_series_to_mode(27295, M)  # TEMP, CHANGE THIS
    
    print(f"index: {i}, star_id: {star_id}, freq: {freq:.4f}")

    # get light curve
    lc, delta_f, sampling_time, exptime = nana.get_kepler_data(star_id)
    t_fit, flux_fit, weight_fit = nana.mask_vals(lc, star_id)

    # periodogram
    over_sampling = 3
    df, f_maxC = delta_f / over_sampling, (3 / (2 * sampling_time))
    f_min = over_sampling * df
    freq_full, power_full = nana.get_periodogram(f_min, f_maxC, df, lc, star_id)

    # get parent and child freqs from db
    conn = nana.get_db_connection()
    cursor = conn.cursor()
    cursor.execute(f"SELECT parent_frequency FROM parent_modes WHERE star_id = '{star_id}'")
    parent_freqs = [row[0] for row in cursor.fetchall()]
    cursor.execute(f"SELECT frequency FROM mode WHERE star_id = '{star_id}' AND parent_mode_id IS NOT NULL")
    child_freqs = [row[0] for row in cursor.fetchall()]
    conn.close()

    # phase fold data
    M = (len(feature) - 1) // 2
    print(f"feature length: {len(feature)}, M: {M}")
    theta = np.linspace(0, 4 * np.pi, 1000)
    yplot0 = np.zeros_like(theta)
    for m in range(1, M + 1):
        a = feature[2*m - 1]
        b = feature[2*m]
        yplot0 += a * np.cos(m * theta) + b * np.sin(m * theta)
    yplot0 += feature[0]
    omega = 2 * np.pi * freq
    phase = (omega * t_fit) % (4 * np.pi)

    # --- subplots ---
    fig, axes = plt.subplots(3,1, figsize=(5,10))

    # light curve
    ax = axes[0]
    ax.plot(t_fit, flux_fit, 'k-', linewidth=0.5, alpha=0.7)
    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Flux")
    ax.set_title(f"Light curve of {star_id}")

    # periodogram
    ax = axes[1]
    ax.plot(freq_full, power_full, 'k.', markersize=1, alpha=0.5)
    ax.set_xlabel("Frequency (1/day)")
    ax.set_ylabel("Power")
    ax.set_yscale('log')
    ax.set_title(f"Log Periodogram of {star_id}")
    for pf in parent_freqs:
        ax.axvline(pf, color='red', alpha=0.6, lw=1)
    for cf in child_freqs:
        ax.axvline(cf, color='red', alpha=0.2, lw=0.5)

    # phase fold
    ax = axes[2]
    ax.plot(phase, flux_fit, alpha=0.1, markersize=2, marker='.', linestyle='none', color='k')
    ax.plot(theta, yplot0, color='r', alpha=0.6, zorder=3)
    ax.set_xlabel('Phase')
    ax.set_ylabel('Flux')
    ax.set_title(rf'Phase fold at freq {freq:.04f} $day^{{-1}}$ for {star_id}')

    plt.tight_layout()
    #plt.savefig('FINAL_pipeline_infographic.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
def phase_folding_by_kic(star_id, M=32):
    star_id = star_id.strip()
    
    #feature, freq = features[i], refined_freqs[i]
    freq, feature = nana.fit_fourier_series_to_mode(27295, M)  # TEMP, CHANGE THIS
    
    #print(f"index: {i}, star_id: {star_id}, freq: {freq:.4f}")

    # get light curve
    lc, delta_f, sampling_time, exptime = nana.get_kepler_data(star_id)
    t_fit, flux_fit, weight_fit = nana.mask_vals(lc, star_id)

    # periodogram
    over_sampling = 3
    df, f_maxC = delta_f / over_sampling, (3 / (2 * sampling_time))
    f_min = over_sampling * df
    freq_full, power_full = nana.get_periodogram(f_min, f_maxC, df, lc, star_id)

    # get parent and child freqs from db
    conn = nana.get_db_connection()
    cursor = conn.cursor()
    cursor.execute(f"SELECT parent_frequency FROM parent_modes WHERE star_id = '{star_id}'")
    parent_freqs = [row[0] for row in cursor.fetchall()]
    cursor.execute(f"SELECT frequency FROM mode WHERE star_id = '{star_id}' AND parent_mode_id IS NOT NULL")
    child_freqs = [row[0] for row in cursor.fetchall()]
    conn.close()

    # phase fold data
    M = (len(feature) - 1) // 2
    print(f"feature length: {len(feature)}, M: {M}")
    theta = np.linspace(0, 4 * np.pi, 1000)
    yplot0 = np.zeros_like(theta)
    for m in range(1, M + 1):
        a = feature[2*m - 1]
        b = feature[2*m]
        yplot0 += a * np.cos(m * theta) + b * np.sin(m * theta)
    yplot0 += feature[0]
    omega = 2 * np.pi * freq
    phase = (omega * t_fit) % (4 * np.pi)

    # --- subplots ---
    fig, axes = plt.subplots(3, 1, figsize=(5, 10))

    # light curve
    ax = axes[0]
    ax.plot(t_fit, flux_fit, 'k-', linewidth=0.5, alpha=0.7)
    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Flux")
    ax.set_title(f"Light curve of {star_id}")

    # periodogram
    ax = axes[1]
    ax.plot(freq_full, power_full, 'k.', markersize=1, alpha=0.5)
    ax.set_xlabel("Frequency (1/day)")
    ax.set_ylabel("Power")
    ax.set_yscale('log')
    ax.set_title(f"Log Periodogram of {star_id}")
    for pf in parent_freqs:
        ax.axvline(pf, color='red', alpha=0.6, lw=1)
    for cf in child_freqs:
        ax.axvline(cf, color='red', alpha=0.2, lw=0.5)

    # phase fold
    ax = axes[2]
    ax.plot(phase, flux_fit, alpha=0.1, markersize=2, marker='.', linestyle='none', color='k')
    ax.plot(theta, yplot0, color='r', alpha=0.6, zorder=3)
    ax.set_xlabel('Phase')
    ax.set_ylabel('Flux')
    ax.set_title(rf'Phase fold at freq {freq:.04f} $day^{{-1}}$ for {star_id}')

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.6)

    # draw arrows between subplots (must be after tight_layout)
    for i in range(2):
        bbox_top = axes[i].get_position()
        bbox_bot = axes[i+1].get_position()
        x = 0.5
        y_start = bbox_top.y0 - 0.04   # just below bottom of upper plot
        y_end = bbox_bot.y1 + 0.03     # just above top of lower plot
        fig.patches.append(
            mpatches.FancyArrowPatch(
                (x, y_start), (x, y_end),
                transform=fig.transFigure,
                arrowstyle='->',
                mutation_scale=20,
                color='gray',
                lw=1.5,
            )
        )

    #plt.savefig('poster_child_combined_2026_06_08.png', bbox_inches='tight', dpi=150)
    plt.show()

In [ ]:
phase_folding_by_kic("KIC005878836", 16)